# Part 1: Debug a Failing Ingestion Job

**Time budget: ~45 minutes.**

## The scenario

It's 7:15 AM. You're on-call. Slack pings:

> **[ALERT]** Freshness SLA breach. `raw_loan_application` last updated yesterday. Downstream marts running on stale data. 12 affected dashboards.

You check the ingestion run. It says **SUCCESS**. No exceptions. No retries. Nothing in the error channel. The watermark advanced.

## The ingestion code that ran

```python
def run_ingestion(config):
    df = spark.read.format("jdbc").options(**jdbc_options(config)).load()
    if config["load_type"] == "FULL":
        df.write.mode("overwrite").saveAsTable(config["target_table"])
    else:
        watermark = get_watermark(config["target_table"])
        df.filter(col(config["control_column"]) > watermark) \
          .write.mode("append").saveAsTable(config["target_table"])
    set_watermark(config["target_table"], status="SUCCESS", ts=datetime.now())
```

## The log

```
2026-05-21 05:12:03 [INFO] Starting ingestion for raw_loan_application
2026-05-21 05:12:03 [INFO] Load type: FULL
2026-05-21 05:12:05 [INFO] Source row count: 0
2026-05-21 05:12:05 [INFO] Writing to raw_loan_application (mode=overwrite)
2026-05-21 05:12:06 [INFO] Watermark updated: status=SUCCESS
2026-05-21 05:12:06 [INFO] Run complete. Rows written: 0
```

The source DB actually has ~50K rows. Yesterday's run was healthy.

## Your tasks

1. Run the setup cell below. It creates a local Delta table that reproduces the failure.
2. **Diagnose** what happened. There's more than one problem with the framework.
3. **Recover** the table. Downstream consumers need data now.
4. **Prevent** this from happening again. Rewrite `run_ingestion` with the safeguards you'd want.

There are dedicated cells below for each task. Add cells freely. Comments and short paragraphs explaining your reasoning are as important as the code.

---
## Setup (run this once)

In [ ]:
# =============================================================================
# SETUP: Creates a local Delta table that reproduces the production failure.
# Idempotent - safe to re-run. Takes ~30 seconds first time, less on re-runs.
# =============================================================================

import os, sys, shutil, random
from contextlib import contextmanager
from datetime import datetime, timedelta, date

# Force Spark to bind to localhost - avoids hostname resolution issues
# in Codespaces / Docker / Linux containers.
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType,
    DateType, TimestampType,
)
from delta import configure_spark_with_delta_pip

WAREHOUSE = os.path.abspath("../spark-warehouse")
DB = "case_study"
TABLE = "raw_loan_application"
FQN = f"{DB}.{TABLE}"

# Hard reset: drop everything that previous runs might have left behind.
if os.path.exists(WAREHOUSE):
    shutil.rmtree(WAREHOUSE)
for path in ["metastore_db", "derby.log"]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

builder = (
    SparkSession.builder.appName("case-study-part1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", WAREHOUSE)
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# ---------------------------------------------------------------------------
# Helper: tolerates a known cosmetic quirk between Delta and Spark's local
# InMemoryCatalog. saveAsTable("overwrite") on a partitioned Delta table
# commits the data fine, but the post-commit catalog-refresh hook asserts.
# This is local-only - real Databricks uses a Hive/Unity Catalog and doesn't
# hit it. We silence the noisy stack trace and verify the data state.
# ---------------------------------------------------------------------------
@contextmanager
def _quiet_stderr():
    fd = sys.stderr.fileno()
    saved = os.dup(fd)
    devnull = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, fd)
        yield
    finally:
        os.dup2(saved, fd)
        os.close(devnull)
        os.close(saved)

def _safe_overwrite(df, table_name, partition_cols=None):
    writer = df.write.format("delta").mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    with _quiet_stderr():
        try:
            writer.saveAsTable(table_name)
        except Exception as e:
            if "AssertionError" not in str(e):
                raise

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")
spark.sql(f"USE {DB}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
spark.sql(f"DROP TABLE IF EXISTS pipeline_watermark")

# --- Build the production-like history ---
random.seed(42)
schema = StructType([
    StructField("application_id",    IntegerType()),
    StructField("customer_id",       IntegerType()),
    StructField("country",           StringType()),
    StructField("amount_eur",        DoubleType()),
    StructField("status",            StringType()),
    StructField("created_date",      DateType()),
    StructField("modified_at",       TimestampType()),
])

def gen_rows(start_id, count, base_day):
    countries = ["EE", "FI", "DE", "ES", "NL"]
    statuses  = ["NEW", "APPROVED", "REJECTED", "PAID"]
    rows = []
    for i in range(count):
        d = base_day - timedelta(days=random.randint(0, 89))
        rows.append((
            start_id + i,
            random.randint(10_000, 99_999),
            random.choice(countries),
            round(random.uniform(500, 25_000), 2),
            random.choices(statuses, weights=[0.20, 0.45, 0.25, 0.10])[0],
            d,
            datetime.combine(d, datetime.min.time()) + timedelta(hours=random.randint(0, 23)),
        ))
    return rows

TODAY = date(2026, 5, 21)

# v0: baseline of 50K rows from prior healthy state
df_v0 = spark.createDataFrame(gen_rows(1, 50_000, TODAY - timedelta(days=1)), schema=schema)
_safe_overwrite(df_v0, FQN, ["created_date"])

# v1: yesterday's incremental added 2K new applications
df_v1 = spark.createDataFrame(gen_rows(50_001, 2_000, TODAY - timedelta(days=1)), schema=schema)
_safe_overwrite(df_v0.union(df_v1), FQN, ["created_date"])

# v2: THE BUG. Today's run wrote 0 rows because the source query returned nothing.
#     The framework happily overwrote the table with the empty result.
df_empty = spark.createDataFrame([], schema=schema)
_safe_overwrite(df_empty, FQN, ["created_date"])

# Watermark table - records the run as SUCCESS even though 0 rows landed.
wm = spark.createDataFrame(
    [("raw_loan_application", "SUCCESS", datetime(2026, 5, 21, 5, 12, 6), 0)],
    ["target_table", "status", "run_at", "rows_written"],
)
_safe_overwrite(wm, "pipeline_watermark")

# --- Verify post-conditions ---
row_count = spark.sql(f"SELECT COUNT(*) FROM {FQN}").collect()[0][0]
history_n = spark.sql(f"DESCRIBE HISTORY {FQN}").count()
assert row_count == 0, f"Expected 0 rows after empty overwrite, got {row_count}"
assert history_n >= 3, f"Expected >=3 Delta versions, got {history_n}"

print("\n" + "=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"\nTable        : {FQN}")
print(f"Warehouse    : {WAREHOUSE}")
print(f"\nCurrent state of raw_loan_application:")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {FQN}").show()
print("Delta history (read this carefully):")
spark.sql(f"DESCRIBE HISTORY {FQN}").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

---
## Task 1: Diagnose

Use the cells below to investigate. **Tell us what happened, why it happened, and what the framework got wrong.** Treat this like a postmortem - your reasoning is the deliverable, not just the code.

Hints (use only if you need them):
- `DESCRIBE HISTORY <table>` shows every version.
- `spark.read.format("delta").option("versionAsOf", N).table("...")` reads a past version.
- There's more than one thing wrong with the framework. Find as many as you can.

In [ ]:
# Your investigation here. Add cells freely.


---
## Task 2: Recover

Downstream consumers need data now. Restore the table to a usable state. Briefly justify your approach (one-time fix vs. something more careful, what to do with the watermark, etc.).

Hints (use if stuck):
- Delta has a purpose-built recovery command. Look up `RESTORE TABLE`.
- Don't forget the watermark. What should it say after recovery, and who should know about the fix?

In [ ]:
# Your recovery code here.


---
## Task 3: Prevent

Rewrite the ingestion function so this class of failure cannot pass silently again. We're looking for the safeguards a defensive framework should have, not a perfect implementation - 20-40 lines of code with comments explaining the reasoning is the right size.

In [ ]:
def run_ingestion_v2(config):
    """Your defensive version. Explain your choices in comments."""
    pass
